In [ ]:
import sys, json, time, platform
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
import river, shap, sklearn
from fraudxai import config, data, cleansing, features, drift, metrics, stream, models, detectors, tuning

FIG = ROOT / "results" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
RES_FILE = ROOT / "results" / "results.json"
R = {}                                              # every number reported in Chapter 3
def save_results():
    RES_FILE.write_text(json.dumps(R, indent=2, default=float))

plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif", "Times New Roman"],
                     "font.size": 10, "axes.titlesize": 10, "figure.dpi": 110, "savefig.dpi": 300,
                     "savefig.bbox": "tight", "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white", "axes.spines.top": False, "axes.spines.right": False})
C_LEGIT, C_FRAUD = "#2E7D32", "#C62828"
pd.set_option("display.width", 140, "display.max_columns", 20, "display.max_colwidth", 90)
R["environment"] = {"python": platform.python_version(), "river": river.__version__, "shap": shap.__version__,
                    "scikit_learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__}
print(R["environment"])

In [ ]:
path = data.download()
raw = data.load(path)
n, n_fraud = len(raw), int(raw.Class.sum())
R["dataset"] = {"source": "OpenML 1597 (mirror of Kaggle mlg-ulb/creditcardfraud)", "file_md5": data.md5(path),
                "rows": n, "columns": raw.shape[1], "fraud": n_fraud, "legit": n - n_fraud,
                "fraud_rate_pct": round(100 * n_fraud / n, 4),
                "time_span_hours": float(raw.Time.max() / 3600),
                "matches_published": (n == config.EXPECTED_ROWS) and (n_fraud == config.EXPECTED_FRAUD)}
save_results(); pd.Series(R["dataset"])

In [ ]:
# Show the loaded dataset
print("Dataset loaded from:", path)
print("Shape (rows, columns):", raw.shape)
print("Fraud cases:", int(raw.Class.sum()), "| Legitimate:", int((raw.Class == 0).sum()))
raw.head(10)

In [ ]:
desc_cols = ["Time", "Amount", "V17", "V14", "V12", "V10"]
desc = raw.groupby("Class")[desc_cols].describe().T.unstack(level=1)
tbl14 = raw[desc_cols].describe().T[["mean", "std", "min", "50%", "max"]]
tbl14["mean_legit"] = raw[raw.Class == 0][desc_cols].mean()
tbl14["mean_fraud"] = raw[raw.Class == 1][desc_cols].mean()
tbl14 = tbl14.round(2)
R["eda_descriptive"] = tbl14.to_dict(orient="index")
tbl14

In [ ]:
counts = raw.Class.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5.2, 3.0))
bars = ax.bar(["Legitimate (0)", "Fraud (1)"], counts.values, color=[C_LEGIT, C_FRAUD], width=0.55)
ax.set_yscale("log"); ax.set_ylabel("Transactions (log scale)")
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v * 1.15, f"{v:,}\n({100*v/n:.3f}%)", ha="center", fontsize=9)
ax.set_ylim(1, counts.max() * 8)
fig.savefig(FIG / "fig09_class_balance.png"); plt.show()
R["eda_class_balance"] = {"legit": int(counts[0]), "fraud": int(counts[1]),
                          "imbalance_ratio_legit_per_fraud": round(counts[0] / counts[1], 1)}

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.2))
amt = [raw.loc[raw.Class == c, "Amount"] + 1 for c in (0, 1)]
bp = ax.boxplot(amt, tick_labels=["Legitimate (0)", "Fraud (1)"], widths=0.5, patch_artist=True,
                flierprops=dict(marker=".", markersize=2, alpha=0.3))
for patch, col in zip(bp["boxes"], [C_LEGIT, C_FRAUD]):
    patch.set_facecolor(col); patch.set_alpha(0.55)
ax.set_yscale("log"); ax.set_ylabel("Amount + 1 (EUR, log scale)")
fig.savefig(FIG / "fig10_amount_by_class.png"); plt.show()
amt_stats = raw.groupby("Class").Amount.agg(["median", "mean", "max", lambda s: (s == 0).sum()])
amt_stats.columns = ["median", "mean", "max", "zero_amount_count"]
R["eda_amount_by_class"] = amt_stats.round(2).to_dict(orient="index")
amt_stats.round(2)

In [ ]:
hours = (raw.Time // 3600).astype(int)                # elapsed hour 0-47 (two days)
per_hour = raw.groupby([hours, "Class"]).size().unstack(fill_value=0)
rate = 100 * per_hour[1] / per_hour.sum(axis=1)
fig, ax1 = plt.subplots(figsize=(6.2, 3.2))
ax1.bar(per_hour.index, per_hour[0], color=C_LEGIT, alpha=0.6, label="Legitimate per hour")
ax1.set_xlabel("Elapsed hour since first transaction (0-47)"); ax1.set_ylabel("Legitimate transactions")
ax2 = ax1.twinx(); ax2.spines["right"].set_visible(True)
ax2.plot(per_hour.index, rate, color=C_FRAUD, marker="o", ms=2.5, lw=1.2, label="Fraud rate (%)")
ax2.set_ylabel("Fraud rate (%)")
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper center", fontsize=8, frameon=False, ncol=2)
fig.savefig(FIG / "fig11_transactions_per_hour.png"); plt.show()
R["eda_per_hour"] = {"min_tx_per_hour": int(per_hour.sum(axis=1).min()), "max_tx_per_hour": int(per_hour.sum(axis=1).max()),
                     "max_fraud_rate_pct": round(float(rate.max()), 3), "hour_of_max_fraud_rate": int(rate.idxmax()),
                     "mean_fraud_rate_pct": round(float(rate.mean()), 3)}
R["eda_per_hour"]

In [ ]:
corr = raw.drop(columns="Time").corr()["Class"].drop("Class")
top = corr.abs().sort_values(ascending=False).head(10).index.tolist()
cm = raw[top + ["Class"]].corr()
fig, ax = plt.subplots(figsize=(5.6, 4.6))
im = ax.imshow(cm.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cm)), cm.columns, rotation=90, fontsize=8); ax.set_yticks(range(len(cm)), cm.columns, fontsize=8)
for i in range(len(cm)):
    for j in range(len(cm)):
        ax.text(j, i, f"{cm.values[i, j]:.2f}", ha="center", va="center", fontsize=6)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pearson r")
fig.savefig(FIG / "fig12_correlation_heatmap.png"); plt.show()
R["eda_corr_with_class_top10"] = corr[top].round(3).to_dict()
corr[top].round(3)

In [ ]:
save_results()

In [ ]:
audit = cleansing.audit_table(raw)
R["data_audit"] = {"columns": len(audit), "total_missing": int(audit.missing.sum()),
                   "dtypes": audit.dtype.value_counts().to_dict()}
audit.round(2).head(8)

In [ ]:
# Validation rules R1-R9 on the raw table (same rules as the unit-tested module)
t = raw.Time
checks = {
    "R1 schema: 31 expected columns": len(set(config.RAW_COLS) ^ set(raw.columns)),
    "R2 all features numeric": int((~raw.dtypes.map(pd.api.types.is_numeric_dtype)).sum()),
    "R3 no missing values":    int(raw.isna().sum().sum()),
    "R4 Class in {0, 1}":      int((~raw.Class.isin([0, 1])).sum()),
    "R5 0 <= Time <= 172,800 s and sorted":
        int(((t < 0) | (t > config.MAX_TIME_S)).sum() + (t.diff() < 0).sum()),
    "R6 Amount >= 0":          int((raw.Amount < 0).sum()),
    "R7 V1-V28, Amount finite": int((~np.isfinite(raw[config.V_COLS + ["Amount"]])).sum().sum()),
    "R8 exact duplicate rows": int(raw.duplicated().sum()),
    "R9 Amount == 0":          int((raw.Amount == 0).sum()),
}
action = ["stop if > 0"] * 4 + ["sort by Time", "stop if > 0", "stop if > 0",
                                "drop, keep first", "keep, flag"]
rules = pd.DataFrame({"violations": list(checks.values()), "action": action}, index=list(checks))
assert rules.violations.tolist() == [r.violations for r in cleansing.validate(raw)]
rules

In [ ]:
# Cleansing: stable sort by Time, drop exact duplicates (keep first), no imputation or capping
df = raw.sort_values("Time", kind="stable").drop_duplicates(keep="first").reset_index(drop=True)
clean_log = {"rows_before": len(raw), "fraud_before": int(raw.Class.sum()),
             "rows_after": len(df), "fraud_after": int(df.Class.sum()),
             "duplicates_dropped": len(raw) - len(df), "fraud_duplicates_dropped": int(raw.Class.sum() - df.Class.sum())}
assert df.equals(cleansing.clean(raw)[0])            # same result as the tested module
R["validation_rules"] = rules.reset_index(names="rule").to_dict(orient="records"); R["cleansing"] = clean_log
save_results(); pd.Series(clean_log)

In [ ]:
# Look-back features: each value uses the current and EARLIER transactions only
t, amt = df.Time.to_numpy(), df.Amount.to_numpy()
fe = df.copy()
fe["hour"] = (t // 3600).astype(int) % 24             # hour of day from Time (s)
fe["log_amount"] = np.log1p(amt)                      # reduces the skew of Amount
# stream velocity: earlier transactions in the previous 60 seconds
fe["tx_count_60s"] = np.arange(len(t)) - np.searchsorted(t, t - 60, side="right")
# mean Amount of the previous 100 transactions, and the current Amount relative to it
prev = pd.Series(amt).shift(1).rolling(100, min_periods=1).mean()
fe["amt_mean_100"] = prev.fillna(0.0).to_numpy()
fe["amt_ratio"] = amt / (fe["amt_mean_100"] + 1.0)
same = np.allclose(fe[features.ENGINEERED], features.add_features(df)[features.ENGINEERED])
print("matches tested module features.add_features:", same)
fe[features.ENGINEERED].describe().T.round(3)

In [ ]:
eng = fe[features.ENGINEERED + ["Class"]]
tbl17 = eng.groupby("Class")[features.ENGINEERED].mean().T.round(3)
tbl17.columns = ["mean_legit", "mean_fraud"]
R["features"] = {"model_features": features.MODEL_FEATURES, "n_model_features": len(features.MODEL_FEATURES),
                 "engineered_means_by_class": tbl17.to_dict(orient="index")}
# check that the online (one-at-a-time) builder gives the same values as the batch builder
online = features.OnlineFeatures(); sample = df.iloc[:5000].to_dict("records")
on = pd.DataFrame([online.transform_one(x) for x in sample])[features.ENGINEERED]
diff = float((on - fe.iloc[:5000][features.ENGINEERED].reset_index(drop=True)).abs().max().max())
R["features"]["online_vs_batch_max_abs_diff_first_5000"] = float(diff)
save_results(); print("max |online - batch| on first 5,000 rows:", diff); tbl17

In [ ]:
n_warm = int(len(fe) * config.WARMUP_FRACTION)
warm, evalp = fe.iloc[:n_warm].reset_index(drop=True), fe.iloc[n_warm:].reset_index(drop=True)
t0 = len(evalp) // 2
DELAYS = {"none (0 s)": 0, "1 hour": 3600, "6 hours": 21600}
# label backlog: number of earlier transactions whose label is still unknown (inside the delay window)
t_eval = evalp.Time.to_numpy(); idx = np.arange(len(t_eval))
backlog = {k: float(np.mean(idx - np.searchsorted(t_eval, t_eval - d, side="right"))) if d else 0.0
           for k, d in DELAYS.items()}
R["stream"] = {"warmup_rows": n_warm, "warmup_fraud": int(warm.Class.sum()), "warmup_end_hour": float(warm.Time.iloc[-1] / 3600),
               "eval_rows": len(evalp), "eval_fraud": int(evalp.Class.sum()), "drift_t0_in_eval": int(t0),
               "drift_t0_hour": float(evalp.Time.iloc[t0] / 3600), "delays_s": DELAYS,
               "mean_unlabelled_backlog": backlog,
               "drift_pool_matches_warmup_top8": set(drift.TOP_FEATURES) == set(warm.drop(columns="Time")[config.V_COLS + ["Class"]].corr()["Class"].drop("Class").abs().nlargest(8).index)}
pd.Series(R["stream"])

In [ ]:
scen = []
for kind in ["abrupt", "gradual", "adversarial"]:
    _, info = drift.inject(evalp, kind, t0=t0, seed=0)
    scen.append({"scenario": kind, **{k: v for k, v in info.items() if k not in ("kind", "seed")}})
warm_top8 = warm.drop(columns="Time")[config.V_COLS + ["Class"]].corr()["Class"].drop("Class").abs().nlargest(8).index.tolist()
R["drift_scenarios_seed0"] = scen; R["drift_feature_pool_from_warmup"] = warm_top8
save_results(); print("top-8 |corr| features in warm-up:", warm_top8); pd.DataFrame(scen)

In [ ]:
FEATS = features.MODEL_FEATURES
warm_rows, y_w, t_w = stream.to_rows(warm, FEATS), warm.Class.to_numpy(), warm.Time.to_numpy()
base = []
for name in ["HT", "LR", "ARF"]:
    for seed in (config.SEEDS if name == "ARF" else [0]):
        t_start = time.time()
        res = stream.prequential(models.make_online(name, seed=seed), warm_rows, y_w, t_w)
        base.append({"model": name, "seed": seed, "seconds": time.time() - t_start, **metrics.summarise(y_w, res.scores)})
base = pd.DataFrame(base)
tbl_base = base.groupby("model")[["pr_auc", "g_mean", "precision_at_k", "seconds"]].agg(["mean", "std"]).fillna(0.0).round(4)
tbl_base["runs"] = base.groupby("model").size()          # HT and LR are deterministic: SD = 0 by construction
R["baselines_warmup_default_params"] = {m: {f"{a}_{b}": float(v) for (a, b), v in row.items()}
                                        for m, row in tbl_base.iterrows()}
save_results(); tbl_base

In [ ]:
# Prequential (test-then-train) grid search on the warm-up slice; 5 seeds per setting
for m, space in tuning.GRID.items():
    print(m, space)
t_start = time.time()
runs = tuning.search(warm[FEATS].to_numpy(), FEATS, y_w, t_w, workers=2)  # parallel runs
summ = tuning.summary(runs)                                     # mean and SD over seeds
best = summ.loc[summ.groupby("model").pr_auc_mean.idxmax()]     # select on PR-AUC
fmt = lambda m: best[m + "_mean"].map("{:.3f}".format) + " ± " + best[m + "_std"].map("{:.3f}".format)
table20 = pd.DataFrame({"model": best.model, "best setting": best.params, "PR-AUC": fmt("pr_auc"),
                        "G-mean": fmt("g_mean"), "P@k": fmt("precision_at_k")}).set_index("model")
R["tuning_runtime_s"] = time.time() - t_start
R["tuning_grid"] = tuning.GRID
R["tuning_all"] = summ.round(4).to_dict(orient="records")
R["tuning_best"] = best.drop(columns=["seconds_mean", "seconds_std"]).round(4).to_dict(orient="records")
save_results(); table20

In [ ]:
summ.round(4)   # every configuration (Table 20 source)

In [ ]:
arf = runs[runs.model == "ARF"]
fig, ax = plt.subplots(figsize=(5.6, 3.2))
for gp, mk, col in [(50, "o", "#1565C0"), (200, "s", "#EF6C00")]:
    g = arf[arf.p_grace_period == gp].groupby("p_n_models").pr_auc.agg(["mean", "std"])
    ax.errorbar(g.index, g["mean"], yerr=g["std"], marker=mk, color=col, capsize=3, lw=1.2, label=f"grace_period = {gp}")
ax.set_xlabel("ARF ensemble size (n_models)"); ax.set_ylabel("Prequential PR-AUC (mean ± SD, 5 seeds)")
ax.set_xticks(sorted(arf.p_n_models.unique())); ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG / "fig13_tuning_curve.png"); plt.show()

In [ ]:
import ast
best_cfg = [(r["model"], ast.literal_eval(r["params"])) for r in R["tuning_best"]]
y_e, t_e = evalp.Class.to_numpy(), evalp.Time.to_numpy()
t_start = time.time()
ev = tuning.evaluate(evalp[FEATS].to_numpy(), FEATS, y_e, t_e, best_cfg, delays=(0.0, 21600.0), workers=2)
ev_s = ev.groupby(["model", "delay_s"])[["pr_auc", "g_mean", "precision_at_k", "seconds"]].agg(["mean", "std"]).fillna(0.0).round(4)
ev_s["runs"] = ev.groupby(["model", "delay_s"]).size()
# frozen batch Random Forest trained on the warm-up slice (O3 comparator)
rf_rows = []
for seed in config.SEEDS:
    rf = models.make_batch_rf(seed=seed).fit(warm[FEATS].to_numpy(), y_w)
    rf_rows.append({"seed": seed, **metrics.summarise(y_e, rf.predict_proba(evalp[FEATS].to_numpy())[:, 1])})
rf_df = pd.DataFrame(rf_rows)
R["preview_eval_online"] = {f"{m}|delay={int(d)}s": {f"{a}_{b}": float(v) for (a, b), v in row.items()} for (m, d), row in ev_s.iterrows()}
R["preview_eval_batch_rf_frozen"] = rf_df[["pr_auc", "g_mean", "precision_at_k"]].agg(["mean", "std"]).round(4).to_dict()
R["preview_eval_runtime_s"] = time.time() - t_start
save_results(); display(ev_s); rf_df[["pr_auc", "g_mean", "precision_at_k"]].agg(["mean", "std"]).round(4)

In [ ]:
from fraudxai import experiments
det = pd.DataFrame(experiments.run_all(warm, evalp, FEATS, t0, {"0 s": 0, "1 h": 3600, "6 h": 21600}, workers=2))
det_s = det.groupby(["scenario", "delay", "detector"]).agg(
    detection_rate=("detected", "mean"), delay_mean=("detection_delay_tx", "mean"), delay_sd=("detection_delay_tx", "std"),
    false_alarms_mean=("false_alarms_before_t0", "mean"), rf_pr_auc_post=("rf_pr_auc_post", "mean")).round(3).reset_index()
R["preview_detectors"] = det_s.to_dict(orient="records")
R["preview_detectors_settings"] = {"horizon_tx": 50000, "shap_delta": 0.002, "adwin_delta": 0.002, "shap_sample_every": 5,
                                   "reference_rows": 2000, "rf": "n_estimators=100, max_depth=10, balanced_subsample"}
save_results(); det_s

In [ ]:
import sys, ast, json, pickle, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
from fraudxai import config, data, cleansing, features, models, stream
R = json.loads((ROOT / "results" / "results.json").read_text())
df_s, _ = cleansing.clean(data.load(data.download()))
warm_s = features.add_features(df_s.iloc[: int(len(df_s) * config.WARMUP_FRACTION)].reset_index(drop=True))
F, SEED = features.MODEL_FEATURES, 0
rf_s = models.make_batch_rf(seed=SEED).fit(warm_s[F].to_numpy(), warm_s.Class.to_numpy())
ref_s = warm_s[F].to_numpy()[np.random.default_rng(SEED).choice(len(warm_s), 2000, replace=False)]
arf_p = ast.literal_eval(next(r["params"] for r in R["tuning_best"] if r["model"] == "ARF"))
arf_s = models.make_online("ARF", seed=SEED, **arf_p)
stream.prequential(arf_s, stream.to_rows(warm_s, F), warm_s.Class.to_numpy(), warm_s.Time.to_numpy())
out = ROOT / "results" / "models"; out.mkdir(parents=True, exist_ok=True)
bundle = {"seed": SEED, "features": F, "rf": rf_s, "shap_reference": ref_s, "arf": arf_s, "arf_params": arf_p,
          "trained_on": f"warm-up slice, {len(warm_s):,} transactions", "saved": time.strftime("%Y-%m-%d %H:%M %Z")}
(out / "fraud_pipeline.pkl").write_bytes(pickle.dumps(bundle))
print("saved", out / "fraud_pipeline.pkl", f"{(out / 'fraud_pipeline.pkl').stat().st_size / 1e6:.1f} MB", "| ARF params:", arf_p)